# Parte individual: grafo de colaboración de Harrison Ford

**Análisis y Ciencia de Datos — Proyecto final**
Herman Echeverría

El notebook corre de principio a fin **sin API key**, leyendo solo de `cache/`.
La key solo se usa para llenar el cache la primera vez: se lee de la variable de entorno
`TMDB_API_KEY` y nunca se escribe en el notebook.

In [ ]:
import os
import re
import json
import time
import itertools
from collections import deque, Counter, defaultdict
from pathlib import Path

import numpy as np
import requests
import matplotlib.pyplot as plt
import networkx as nx            # solo para visualizar y verificar
from sklearn.cluster import KMeans

In [ ]:
# Configuración
CACHE_DIR = Path("cache")
BASE_URL = "https://api.themoviedb.org/3"

SEED_ID = 3                      # Harrison Ford (verificado con GET /person/3)
SEED_NAME = "Harrison Ford"
LIMIT = 5
START_DATE = "1975-01-01"
END_DATE = "2025-12-31"

API_KEY = os.environ.get("TMDB_API_KEY")   # None si no hay key: entonces solo se lee el cache

## I.2 Clase `Graph`

Grafo **no dirigido y con pesos**. Cada arista se guarda con la llave `(min(a,b), max(a,b))`,
así $(a,b)$ y $(b,a)$ son la misma arista.

In [ ]:
class Graph:
    def __init__(self):
        self.nodes = {}   # id -> name
        self.edges = {}   # (id menor, id mayor) -> peso

    @staticmethod
    def _key(a, b):
        return (a, b) if a <= b else (b, a)

    def add_node(self, node):
        """node es una tupla (id, name). La agrega solo si no existe."""
        node_id, name = node
        if node_id not in self.nodes:
            self.nodes[node_id] = name

    def add_edge(self, a, b):
        """Agrega la arista entre los id a y b con peso 1, o le suma 1 si ya existe."""
        if a == b:
            return
        key = self._key(a, b)
        self.edges[key] = self.edges.get(key, 0) + 1

    def total_nodes(self):
        return len(self.nodes)

    def total_edges(self):
        return len(self.edges)

    def degree(self, node_id):
        """Número de aristas que tocan al nodo (no la suma de pesos)."""
        return sum(1 for a, b in self.edges if node_id in (a, b))

    def degrees(self):
        deg = {n: 0 for n in self.nodes}
        for a, b in self.edges:
            deg[a] += 1
            deg[b] += 1
        return deg

    def max_degree_nodes(self):
        """Diccionario {id: grado} con el nodo (o los nodos, si hay empate) de grado máximo."""
        deg = self.degrees()
        if not deg:
            return {}
        m = max(deg.values())
        return {n: d for n, d in deg.items() if d == m}

    def adjacency(self):
        """Devuelve (A, ids): A es la matriz de adyacencia con pesos (numpy) e ids el orden de sus filas."""
        ids = list(self.nodes)
        pos = {n: i for i, n in enumerate(ids)}
        A = np.zeros((len(ids), len(ids)))
        for (a, b), w in self.edges.items():
            A[pos[a], pos[b]] = w
            A[pos[b], pos[a]] = w
        return A, ids

In [ ]:
# Pruebas pequeñas de la clase
g_test = Graph()
g_test.add_node((1, "A")); g_test.add_node((2, "B")); g_test.add_node((3, "C"))
g_test.add_node((1, "A repetido"))
g_test.add_edge(1, 2); g_test.add_edge(2, 1); g_test.add_edge(2, 3)
assert g_test.total_nodes() == 3 and g_test.nodes[1] == "A"
assert g_test.total_edges() == 2 and g_test.edges[(1, 2)] == 2
assert g_test.max_degree_nodes() == {2: 2}
A_test, ids_test = g_test.adjacency()
assert np.allclose(A_test, A_test.T) and A_test[0, 1] == 2
print("Graph OK")

## I.3 Clase `TMDBAPIUtils`

Cada respuesta del API se guarda como JSON en `cache/people/<person_id>.json` y
`cache/movies/<movie_id>.json`, y se lee de ahí si ya existe.

In [ ]:
SELF_ROLE = re.compile(r"\b(him|her|them)?self\b", re.IGNORECASE)


class TMDBAPIUtils:
    def __init__(self, api_key=None, cache_dir=CACHE_DIR):
        self.api_key = api_key
        self.cache_dir = Path(cache_dir)

    def _get(self, endpoint, cache_file):
        """Lee cache_file si existe; si no, llama al API (endpoint), guarda la respuesta y la devuelve."""
        cache_file = Path(cache_file)
        if cache_file.exists():
            return json.loads(cache_file.read_text(encoding="utf-8"))
        if not self.api_key:
            raise RuntimeError(f"{cache_file} no está en el cache y no hay TMDB_API_KEY")
        for intento in range(5):
            try:
                r = requests.get(BASE_URL + endpoint,
                                 params={"api_key": self.api_key, "language": "en-US"}, timeout=30)
            except requests.RequestException as e:
                # sin "from e": el mensaje original trae la URL con la API key
                raise RuntimeError(f"Fallo de red pidiendo {endpoint}: {type(e).__name__}") from None
            if r.status_code == 429:          # demasiadas solicitudes: esperar y reintentar
                time.sleep(2 ** intento)
                continue
            if not r.ok:
                raise RuntimeError(f"TMDB respondió {r.status_code} para {endpoint}")
            break
        data = r.json()
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        cache_file.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        return data

    def get_movie_cast(self, movie_id, limit=None, exclude_ids=None):
        """Elenco de la película: solo miembros con 0 <= order <= limit - 1 y cuyo id no esté en exclude_ids."""
        data = self._get(f"/movie/{movie_id}/credits", self.cache_dir / "movies" / f"{movie_id}.json")
        exclude = set(exclude_ids or [])
        cast = []
        for c in data.get("cast", []):
            if c["order"] < 0 or (limit is not None and c["order"] > limit - 1):
                continue
            if c["id"] in exclude:
                continue
            cast.append({"id": c["id"], "name": c["name"], "order": c["order"],
                         "character": c.get("character", "")})
        return cast

    def get_movie_credits_for_person(self, person_id, start_date=None, end_date=None):
        """Una entrada por película en la que actuó la persona: {id, title, release_date, order}.
        Filtra por fecha, quita documentales (género 99) y papeles donde se interpreta a sí misma."""
        data = self._get(f"/person/{person_id}/movie_credits", self.cache_dir / "people" / f"{person_id}.json")
        movies = {}
        for c in data.get("cast", []):
            rd = c.get("release_date") or ""
            if not rd:                                    # sin fecha no se puede saber si cae en la ventana
                continue
            if (start_date and rd < start_date) or (end_date and rd > end_date):
                continue
            if 99 in c.get("genre_ids", []):
                continue
            if SELF_ROLE.search(c.get("character") or ""):
                continue
            # una persona puede salir dos veces en la misma película: nos quedamos con su mejor order
            if c["id"] not in movies or c["order"] < movies[c["id"]]["order"]:
                movies[c["id"]] = {"id": c["id"], "title": c["title"],
                                   "release_date": rd, "order": c["order"]}
        return list(movies.values())

In [ ]:
tmdb = TMDBAPIUtils(api_key=API_KEY)